In [0]:
# COMMAND ----------
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, explode, current_timestamp
from delta.tables import DeltaTable
from pyspark.sql.functions import from_json, ArrayType

# COMMAND ----------
# 1. Outer Schema Enforcement
circuits_schema = StructType(fields=[
    StructField("MRData", StructType([
        StructField("CircuitTable", StructType([
            StructField("Circuits", StringType(), True)
        ]), True)
    ]), True)
])

raw_circuits_df = spark.read \
    .schema(circuits_schema) \
    .json("dbfs:/mnt/f1-raw/circuits/*")

# COMMAND ----------
# 2. Inner Array Schema Definition & Transformation
circuit_element_schema = StructType([
    StructField("circuitId", StringType(), False),
    StructField("circuitName", StringType(), True),
    StructField("Location", StructType([
        StructField("lat", StringType(), True),
        StructField("long", StringType(), True),
        StructField("locality", StringType(), True),
        StructField("country", StringType(), True)
    ]), True)
])

parsed_df = raw_circuits_df.withColumn(
    "circuit_array", 
    from_json(col("MRData.CircuitTable.Circuits"), ArrayType(circuit_element_schema))
)
exploded_df = parsed_df.select(explode(col("circuit_array")).alias("circuit"))

silver_circuits_df = exploded_df.select(
    col("circuit.circuitId").alias("circuit_id"),
    col("circuit.circuitName").alias("name"),
    col("circuit.Location.locality").alias("location"),
    col("circuit.Location.country").alias("country"),
    col("circuit.Location.lat").cast("double").alias("latitude"),
    col("circuit.Location.long").cast("double").alias("longitude"),
    current_timestamp().alias("ingestion_date")
).dropDuplicates(["circuit_id"])

display(silver_circuits_df)

# COMMAND ----------
# 3. Hive Metastore Compliant Table Write & Upsert
spark.sql("CREATE DATABASE IF NOT EXISTS hive_metastore.f1_transformed")

if not spark.catalog.tableExists("hive_metastore.f1_transformed.circuits"):
    silver_circuits_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("hive_metastore.f1_transformed.circuits")
    print("Circuits table successfully initialized as a Hive Metastore Managed Table.")
else:
    tgt_table = DeltaTable.forName(spark, "hive_metastore.f1_transformed.circuits")
    tgt_table.alias("tgt") \
        .merge(
            source = silver_circuits_df.alias("src"), 
            condition = "tgt.circuit_id = src.circuit_id"
        ) \
        .whenMatchedUpdate(set = {
            "name": "src.name", 
            "location": "src.location", 
            "country": "src.country", 
            "latitude": "src.latitude", 
            "longitude": "src.longitude", 
            "ingestion_date": "src.ingestion_date"
        }) \
        .whenNotMatchedInsert(values = {
            "circuit_id": "src.circuit_id", 
            "name": "src.name", 
            "location": "src.location", 
            "country": "src.country", 
            "latitude": "src.latitude", 
            "longitude": "src.longitude", 
            "ingestion_date": "src.ingestion_date"
        }) \
        .execute()
    print("Circuits incremental update completed successfully via Hive Metastore Delta Merge.")
